# CropCop Track B — Historical Comparison Index Builder

**Infrastructure-only notebook; not a claim-producing evaluation.** Attach exactly one Track-B `core` package and one `historical_source` package covering the complete 117,546-image V4 audited universe. Use Kaggle **T4x2**; the builder uses `cuda:0` and four CPU workers.

The output is the immutable `historical_compare` Kaggle dataset consumed by the final Track-B notebook. It contains SHA/pHash/dHash identity, frozen DINO audit features, and packed ORB representations, not classifier predictions.

## Environment

Configure Kaggle Dependency Manager with `journal_extension/track_b_r07/requirements-trackb.lock.txt` before **Save & Run All**. The output must fit `/kaggle/working` and should be published as a private immutable Kaggle dataset after this builder returns PASS.

In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys
INPUT=Path('/kaggle/input')
OUTPUT=Path('/kaggle/working/cropcop_hist_compare_v1')
core=[]; hist=[]
for p in INPUT.glob('**/TRACKB_INPUT_MANIFEST.json'):
    o=json.loads(p.read_text());
    if o.get('role')=='core': core.append((p,o))
for p in INPUT.glob('**/TRACKB_HIST_SOURCE_MANIFEST.json'):
    o=json.loads(p.read_text());
    if o.get('role')=='historical_source': hist.append((p,o))
if len(core)!=1 or len(hist)!=1: raise RuntimeError(f'Need exactly one core and one historical_source package; got core={len(core)}, historical_source={len(hist)}')
core_p,core_o=core[0]; hist_p,hist_o=hist[0]
repo=(core_p.parent/core_o['repository_root']).resolve()
runner=repo/'journal_extension/scripts/build_trackb_historical_compare.py'
if not runner.is_file(): raise RuntimeError(f'Builder missing: {runner}')
def f(bundle_p,bundle_o,key): return (bundle_p.parent/bundle_o['files'][key]['path']).resolve()
source_manifest=f(hist_p,hist_o,'source_manifest')
image_root=(hist_p.parent/hist_o['image_root']).resolve()
dino=f(core_p,core_o,'dino_checkpoint')
factory=f(core_p,core_o,'dino_factory_manifest')
execution_lock=f(core_p,core_o,'execution_lock')
code_attestation=f(core_p,core_o,'code_attestation')
factory_root=(core_p.parent/core_o['dino_factory_source_root']).resolve()
print('Free working GB:', round(shutil.disk_usage('/kaggle/working').free/1024**3,2))
subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'],check=True)


In [ ]:
if OUTPUT.exists() and any(OUTPUT.iterdir()): raise RuntimeError(f'Output must be empty: {OUTPUT}')
cmd=[sys.executable,str(runner),'--source-manifest',str(source_manifest),'--image-root',str(image_root),'--id-column',hist_o.get('id_column','hist_id'),'--path-column',hist_o.get('path_column','relative_path'),'--dino-checkpoint',str(dino),'--factory-manifest',str(factory),'--factory-source-root',str(factory_root),'--repo-root',str(repo),'--execution-lock',str(execution_lock),'--code-attestation',str(code_attestation),'--output-dir',str(OUTPUT),'--device','cuda:0','--workers','4','--orb-chunk-size','512','--dino-batch-size','64']
print('Launching historical index builder')
subprocess.run(cmd,cwd=repo,check=True)


In [ ]:
manifest=json.loads((OUTPUT/'TRACKB_INPUT_MANIFEST.json').read_text())
cert=json.loads((OUTPUT/'BUILD_CERTIFICATE.json').read_text())
if manifest.get('role')!='historical_compare' or manifest.get('image_count')!=117546 or cert.get('status')!='PASS': raise RuntimeError('Historical comparison package did not seal correctly')
print(json.dumps(cert,indent=2,sort_keys=True))
print('Publish this output directory as an immutable private Kaggle Dataset, then attach that fixed version to the final Track-B notebook.')
